# От фотографий iNaturalist до сравнения четырёх моделей

**Run All** запускает этапы `01 → 02 → 03a → 03b → 03c → 03d → модели` в одном ядре. Обычный запуск повторяет эксперимент команды. Исходные наблюдения — снимок выдачи iNaturalist, ручная разметка и ответы исполнителей. Отбор 400 + 100, загрузка оригиналов, построение эталона, инференс и оценка выполняются заново. Каждая ячейка ниже использует файлы или переменные этого прогона.

Для полного запуска нужны интернет, NVIDIA GPU с CUDA, зависимости инференса и собранный `llama-server`. Вычисления и датасеты пишутся в отдельные каталоги. Если нужна только офлайн-проверка опубликованных результатов, у неё есть отдельный ноутбук. [Подготовка окружения](docs/reproduce.md) · [Проверка сохранённых результатов](notebooks/reproduce_saved_results.ipynb)

## Настройки запуска

- **historical** — воспроизводит исходную выборку по снимку 700 кандидатов и принимает реальные ответы людей как входные данные. Можно сравнить новые предсказания с исходным экспериментом. Платные пулы в этом режиме не создаются.
- **live** — начинает с сегодняшнего API iNaturalist. Новые контрольные примеры размечаются вручную. Ячейки Яндекс Заданий публикуют платные пулы и ждут реальных исполнителей. Для этого режима нужен токен заказчика `YANDEX_TASKS_OAUTH_TOKEN` в окружении или локальном `.env`. Выбирайте его для нового сбора разметки.

`RESUME_RUN = None` начинает новый прогон при каждом Run All. Чтобы продолжить после остановки, впишите идентификатор уже начатого прогона и оставьте его режим. Вычислительные seed одинаковы для всех запусков. Идентификатор определяет только папку на диске.

In [ ]:
MODE = "historical"  # historical | live
RESUME_RUN = None   # например, строка с ID из вывода следующей ячейки
EXTRA_REVIEW_IDS = []  # live: дополнительные фото, выбранные заказчиком на переразметку

In [ ]:
import json
import os
import re
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from tempfile import NamedTemporaryFile

from IPython.display import Markdown, display
from PIL import Image

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").is_file() and (p / "butterfly_bench").is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from butterfly_bench.schema import validate_boxes
from scripts.download_selected import checksum, jsonl_rows
from scripts.pipeline_sampling import collect_candidates, select_photos, download_photos
from scripts.pipeline_models import MODEL_NAMES, prepare_model_manifest, run_model

def indexed(rows):
    by_id = {row["photo_id"]: row for row in rows}
    if len(by_id) != len(rows):
        raise ValueError("Повторяется photo_id")
    return by_id

def write_rows(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile("w", dir=path.parent, encoding="utf-8", delete=False) as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
    Path(stream.name).replace(path)

def table(headers, rows):
    lines = [headers, ["---"] * len(headers), *rows]
    display(Markdown("\n".join("| " + " | ".join(map(str, row)) + " |" for row in lines)))

if MODE not in ("historical", "live"):
    raise ValueError("Выберите historical или live")
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") if RESUME_RUN is None else RESUME_RUN
if not isinstance(run_id, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,90}", run_id):
    raise ValueError("Используйте только буквы, цифры, подчёркивания и дефисы в ID")
run_dir = root / "outputs/pipeline" / run_id
downloads = root.parent / "downloads/final-project"
image_dir = downloads / "pipeline" / run_id / "images"
os.environ.setdefault("HF_HOME", str(downloads / "huggingface"))
plan = json.loads((root / "data/sampling/sampling_plan.json").read_text())
model_archive = root / "results/final_400"
frozen_plan = json.loads((model_archive / "plan.json").read_text())
archive = root / "data/sampling/runs/20261108T001420_435829Z"
metadata = {"mode": MODE, "sampling_seed": plan["seed"], "model_seed": frozen_plan["seed"]}
if (metadata["sampling_seed"], metadata["model_seed"]) != (20261107, 42):
    raise ValueError("Изменились исходные seed проекта")
if RESUME_RUN is None:
    run_dir.mkdir(parents=True)
    (run_dir / "run.json").write_text(json.dumps(metadata, indent=2) + "\n")
elif not (run_dir / "run.json").is_file() or json.loads((run_dir / "run.json").read_text()) != metadata:
    raise ValueError("Для этого ID не найден запуск с тем же режимом и seed")
print("RUN_ID:", run_id, "| режим:", MODE)
print("Результаты:", run_dir, "\nОригиналы:", image_dir)
print("Seed отбора:", plan["seed"], "| seed моделей:", frozen_plan["seed"])

## 01. Получить кандидатов, отобрать фотографии и сохранить оригиналы

Сначала исключаем фотографии отдельного конкурсного набора. В историческом режиме начинаем со всех кандидатов, полученных в ноябре, до назначения в выборки. В режиме live запросы идут в API по тем же таксонам, страницам, фильтру CC0 и seed. Меняется день запроса — может измениться список доступных наблюдений. [План отбора](data/sampling/sampling_plan.json) · [Исходный ноутбук 01](notebooks/data/01_build_photo_manifests.ipynb)

In [ ]:
contest = jsonl_rows(root / "data/contest_400/photos.jsonl")
template = jsonl_rows(root / "data/contest_400/submission-template.jsonl")
def photo_urls(rows):
    return {i: row["photo_url"] for i, row in indexed(rows).items()}
if photo_urls(contest) != photo_urls(template) or len(contest) != 400:
    raise ValueError("Конкурсный манифест и шаблон посылки должны описывать одни и те же 400 фото")

snapshot = archive / "candidates_from_api.jsonl" if MODE == "historical" else None
candidates = collect_candidates(run_dir, plan, contest, snapshot=snapshot)
counts = Counter(r["source_stratum"] for r in candidates)
table(["Группа iNaturalist", "Кандидаты", "Квота для сравнения", "Для материалов исполнителей"],
      [[g["name"], counts[g["name"]], g["target_count"], g["training_exam_count"]] for g in plan["groups"]])

### Выбор 400 + 100

Внутри каждой группы действует первоначальный алгоритм случайного отбора. Он проверяет размеры изображения и использует `seed + 1000 + номер группы`. Сохраняются CSV с назначениями и галерея кандидатов. В историческом режиме сверяем получившиеся наборы по каждому ID и URL до скачивания.

In [ ]:
selected = select_photos(run_dir, candidates, plan)
main_selected = [r for r in selected if r["purpose"] == "main"]
ref_selected = [r for r in selected if r["purpose"] == "worker_train_exam"]
assert len(main_selected) == 400 and len(ref_selected) == 100
assert set(photo_urls(main_selected)).isdisjoint(photo_urls(ref_selected))
assert set(photo_urls(selected)).isdisjoint(photo_urls(contest))
if MODE == "historical":
    assert photo_urls(main_selected) == photo_urls(jsonl_rows(root / "data/golden_400/photos.jsonl"))
    assert photo_urls(ref_selected) == photo_urls(jsonl_rows(root / "data/reference_100/photos.jsonl"))
print(f"Из {len(candidates)} выбраны {len(main_selected)} основных и {len(ref_selected)} вспомогательных фото.")
print("Назначения:", run_dir / "selection_500_proposal.csv")

### Загрузка и проверка

Для нового ID оригиналы загружаются в пустой каталог. При возобновлении уже скачанные файлы проверяются повторно. Проверяем формат, размеры, уникальность SHA-256 и близость изображений по dHash. Исходные URL и местоположение файла записываются в манифесты. Для исторических фото дополнительно проверяем SHA-256 каждого оригинала по журналу первого запуска.

In [ ]:
expected_hashes = None
if MODE == "historical":
    expected_hashes = {r["photo_id"]: r["sha256"] for r in jsonl_rows(archive / "download_audit.jsonl")}
audit = download_photos(run_dir, selected, image_dir, expected_hashes=expected_hashes)
main_path = run_dir / "main/photos.jsonl"
main = jsonl_rows(main_path)
reference_photos = jsonl_rows(run_dir / "reference/photos.jsonl")
print(f"{len(audit)} проверенных оригиналов, {sum(r['bytes'] for r in audit):,} байт")
print(main_path, run_dir / "reference/photos.jsonl", sep="\n")

## 02. Проверить разметку вспомогательной сотни

В исходном проекте эти сто фотографий разметили вручную. Для повтора читаем первичный JSON-массив разметчика, проверяем его связь с только что выбранными фотографиями и сохраняем проверенные рамки. Для новых кадров Jupyter показывает фотографии и принимает координаты. Ответ `none` означает, что человек осмотрел снимок и взрослых бабочек нет. [Исходный ноутбук 02](notebooks/data/02_review_worker_reference.ipynb)

In [ ]:
from scripts.pipeline_sampling import prepare_reference, review_reference, load_reference

if MODE == "historical":
    expert = json.loads((root / "data/reference_100/worker_reference_first_pass.json").read_text())
    if photo_urls(expert) != photo_urls(reference_photos):
        raise ValueError("Фотографии разметчика и выбранная сотня различаются")
    write_rows(run_dir / "reference/labels.jsonl", expert)
else:
    prepare_reference(run_dir)
    review_reference(run_dir)
reference = load_reference(run_dir)
print("Фото:", len(reference), "| с бабочками:", sum(bool(r["boxes"]) for r in reference),
      "| рамок:", sum(len(r["boxes"]) for r in reference))

In [ ]:
from butterfly_bench.visualize import draw_prediction

def show_boxes(path, boxes):
    with Image.open(path) as source:
        annotated = draw_prediction(source.convert("RGB"), {"boxes": boxes, "error": None})
    annotated.thumbnail((600, 450))
    display(annotated)

example = next((r for r in reference if r["boxes"]), reference[0])
photo = indexed(reference_photos)[example["photo_id"]]
print("Пример для разметчика:", photo["photo_id"])
show_boxes(run_dir / "reference" / photo["image_path"], example["boxes"])

## 03a. Экзамен разметчиков и первичный ответ

Исторический режим восстанавливает экзаменационный ключ и рабочие страницы для выбранных ID. Разметка людей сохранена во входах для независимых ревьюеров. Оттуда получаем статус, рамки и идентификатор выдачи каждого первого ответа.

В live создаются проект, навыки и экзамен. Рабочий пул получает выбранные 400 фото, по четыре на страницу. Обработка продолжится после ответов людей. [Исходный ноутбук 03a](notebooks/crowd/03a_create_annotation.ipynb)

In [ ]:
inputs = root / "crowd"
crowd_dir = run_dir / "crowd"
crowd_dir.mkdir(exist_ok=True)
main_urls = photo_urls(main)

def historical_answers(filename, expected_ids):
    pages = json.loads((inputs / "review/tasks" / filename).read_text())
    rows, controls = [], 0
    for page in pages:
        for task in page["tasks"]:
            if "known_solutions" in task:
                controls += 1
                continue
            v = task["input_values"]
            if v["image"] != main_urls[v["photo_id"]]:
                raise ValueError(f"URL разметки отличается для {v['photo_id']}")
            rows.append({"photo_id": v["photo_id"], "photo_url": v["image"], "status": v["worker_status"],
                         "source_assignment_id": v["source_assignment_id"], "boxes": validate_boxes(json.loads(v["boxes_json"]))})
    records = indexed(rows)
    if records.keys() != set(expected_ids):
        raise ValueError(f"Неполный набор ответов в {filename}")
    print(f"{filename}: {len(pages)} страниц, {len(rows)} рабочих заданий, {controls} контрольных")
    return records

if MODE == "historical":
    expert_by_id = indexed(reference)
    exam = jsonl_rows(inputs / "annotation/exam_key.jsonl")
    for item in exam:
        assert item["boxes"] == expert_by_id[item["photo_id"]]["boxes"]
    write_rows(crowd_dir / "annotation_exam_key.jsonl", exam)
    work_pages = json.loads((inputs / "annotation/tasks/annotation_work.json").read_text())
    tasks = [t["input_values"] for p in work_pages for t in p["tasks"]]
    assert len(tasks) == len(main) and {t["photo_id"]: t["image"] for t in tasks} == main_urls
    print(f"Экзамен — {len(exam)} фото. Первичная разметка — {len(work_pages)} страниц.")
    first = historical_answers("validation_work.json", main_urls)
else:
    from dotenv import load_dotenv
    from scripts.pipeline_crowd import CrowdRun
    load_dotenv(root / ".env")
    crowd = CrowdRun(run_dir, main, reference)
    first = crowd.annotate()
write_rows(crowd_dir / "first_answers.jsonl", first.values())
print("Статусы первой разметки:", dict(Counter(r["status"] for r in first.values())))

## 03b. Независимая проверка

При новом сборе исполнители проходят обучение и экзамен, затем три ревьюера проверяют каждый рабочий ответ. В историческом наборе сохранились итоговые количества голосов, список отправленных на переразметку и задания проверяющих. Для первоначального раунда архив содержит ограниченную информацию: набор 43 сформировали из **26** случаев по ревью и **17** по дополнительному просмотру. Используем этот фактически утверждённый список. [Исходный ноутбук 03b](notebooks/crowd/03b_create_validation.ipynb)

In [ ]:
if MODE == "historical":
    recorded_reviews = indexed(jsonl_rows(root / "data/golden_400/review.jsonl"))
    assert recorded_reviews.keys() == first.keys()
    correction_pages = json.loads((inputs / "annotation/tasks/reannotation.json").read_text())
    correction_tasks = [t["input_values"] for p in correction_pages for t in p["tasks"]]
    correction_ids = {t["photo_id"] for t in correction_tasks}
    assert len(correction_tasks) == len(correction_ids) and correction_ids <= first.keys()
    assert all(main_urls[t["photo_id"]] == t["image"] for t in correction_tasks)
    print("На второй проход по решению команды:", len(correction_ids))
else:
    votes = crowd.review(first)
    print("Вердикты первого прохода:", dict(Counter(v["majority"] for v in votes.values())))

## 03c. Переразметка спорных фотографий

Повторный ответ привязывается к тому же `photo_id`. Сравним координаты до и после. [Исходный ноутбук 03c](notebooks/crowd/03c_create_reannotation.ipynb)

In [ ]:
if MODE == "historical":
    revised = historical_answers("revalidation.json", correction_ids)
else:
    revised = crowd.reannotate(first, votes, extra_ids=EXTRA_REVIEW_IDS)
write_rows(crowd_dir / "second_answers.jsonl", revised.values())
print(f"Получено {len(revised)} повторных ответов, рамки изменились на "
      f"{sum(r['boxes'] != first[i]['boxes'] for i, r in revised.items())} снимках.")

## 03d. Повторная проверка и сборка эталона

В историческом эксперименте заказчик включил все 43 повторных ответа, в том числе 35 с отрицательным большинством. Для точного повтора применяем то же решение и сохраняем голоса отдельно. При новом сборе оставшиеся спорные случаи передаются заказчику на ручное решение. [Исходный ноутбук 03d](notebooks/crowd/03d_create_revalidation.ipynb)

In [ ]:
if MODE == "historical":
    final = first | revised
    provenance = []
    for pid, answer in final.items():
        record = recorded_reviews[pid]
        verdict = record["review"]
        assert sum(verdict["votes"].values()) == 3 and verdict["votes"][verdict["majority"]] >= 2
        assert record["status"] == answer["status"]
        assert (record["annotation_source"] == "reannotation_43") == (pid in revised)
        provenance.append(record)
    write_rows(crowd_dir / "review.jsonl", provenance)
    gold_path = crowd_dir / "golden.jsonl"
    write_rows(gold_path, ({k: final[i][k] for k in ("photo_id", "photo_url", "boxes")} for i in sorted(final)))
    restored = indexed(jsonl_rows(gold_path))
    assert restored == indexed(jsonl_rows(root / "data/golden_400/annotations.jsonl"))
    print("Все рамки и URL совпали с историческим эталоном.")
else:
    gold_path = crowd.finish(first, votes, revised)
gold = indexed(jsonl_rows(gold_path))
assert gold.keys() == first.keys()
print(f"Эталон: {len(gold)} фото, {sum(len(r['boxes']) for r in gold.values())} рамок, "
      f"{sum(not r['boxes'] for r in gold.values())} снимков без рамок.")

## 04. Подать одни и те же фотографии четырём моделям

Локальный вход объединяет скачанный оригинал с рамками из предыдущей ячейки по `photo_id`. В историческом режиме восстанавливаем порядок 400 ID и сверяем пиксели с закреплённым файлом Hugging Face, который читал исходный инференс. Его проверочная копия кешируется. Конфигурации фиксируют ревизии весов, промпты, пороги, NMS и повторы. У всех моделей seed **42**. Обучение не требуется. [Исходное демо моделей](notebooks/models/03_model_demo.ipynb) · [Зафиксированные конфигурации](results/final_400/frozen/)

In [ ]:
from omegaconf import OmegaConf
model_dir = run_dir / "models"
dataset_info = model_archive / "dataset.json"
order = json.loads(dataset_info.read_text())["photo_ids"] if MODE == "historical" else None
manifest = prepare_model_manifest(main_path, gold_path, image_dir, model_dir, photo_ids=order)
for name in MODEL_NAMES:
    path = model_archive / "frozen" / f"{name}.yaml"
    expected = [v for k, v in frozen_plan["sha256"].items() if k.endswith("/frozen/" + path.name)]
    assert expected == [checksum(path)] and OmegaConf.load(path).seed == 42
if MODE == "historical":
    from scripts.pipeline_models import verify_historical_pixels
    pixel_check = verify_historical_pixels(manifest, dataset_info, Path(os.environ["HF_HOME"]) / "hub")
    print("Проверены входы исходного GPU-прогона:", pixel_check)
print("Вход моделей:", manifest, "| фотографий:", len(jsonl_rows(manifest)))

### Реальный инференс

Модели запускаются последовательно на CUDA. Для каждого запуска создаются собственные `predictions.jsonl`, `attempts.jsonl`, метрики и журналы. При возобновлении того же ID используются только завершённые результаты этого прогона с проверенными входами. Новый ID запускает все четыре модели заново.

In [ ]:
results = []
for name in MODEL_NAMES:
    print(f"Запуск {name} на {len(gold)} снимках…", flush=True)
    result = run_model(name, manifest, model_dir)
    results.append(result)
    print(f"{name}: {result['status']}, F1@0.5 = {result['metrics']['f1']:.6f}")

### Пересчитать метрики и выбрать победителя

Сопоставление рамок выполняется отдельно для каждой фотографии при IoU 0,50 и 0,75. Побеждает наибольший object F1 при IoU 0,50. Исторические предсказания читаются только после инференса для сверки. Несовпадение выводится в отчёте: смена GPU, CUDA, версий библиотек или сборки llama.cpp может повлиять на результат даже с тем же seed.

In [ ]:
from butterfly_bench.metrics import evaluate

truth = {i: r["boxes"] for i, r in gold.items()}
summary, predictions = [], {}
for result in results:
    name = result["model"]
    rows = indexed(jsonl_rows(Path(result["output_dir"]) / "predictions.jsonl"))
    assert rows.keys() == truth.keys()
    predictions[name] = rows
    metrics = evaluate(rows, truth)
    assert all(value == result["metrics"][key] for key, value in metrics.items())
    entry = {"model": name, "f1": metrics["f1"], "f1_075": metrics["iou_0.75/f1"],
             "fp": metrics["fp"], "fn": metrics["fn"], "processing_errors": metrics["processing_errors"]}
    if MODE == "historical":
        baseline = indexed(jsonl_rows(model_archive / name / "predictions.jsonl"))
        assert baseline.keys() == rows.keys()
        entry["different_photos"] = sum(row != baseline[i] for i, row in rows.items())
        entry["delta_f1"] = metrics["f1"] - evaluate(baseline, truth)["f1"]
    summary.append(entry)
summary.sort(key=lambda r: -r["f1"])
write_rows(run_dir / "comparison.jsonl", summary)
columns = list(summary[0])
table(columns, [[f"{r[k]:.6f}" if isinstance(r[k], float) else r[k] for k in columns] for r in summary])
winners = [r["model"] for r in summary if r["f1"] == summary[0]["f1"]]
print("Лучший F1@0.5:", ", ".join(winners))

### Посмотреть рамки и исходный ответ

Один и тот же кадр показывается с человеческим эталоном (зелёный) и свежими прогнозами каждой модели (красный). Для диагностики можно выбрать другой ID из манифеста.

In [ ]:
from matplotlib import pyplot as plt

model_inputs = indexed(jsonl_rows(manifest))
example_id = 751308237 if 751308237 in gold else next((i for i, r in gold.items() if r["boxes"]), next(iter(gold)))
with Image.open(manifest.parent / model_inputs[example_id]["image_path"]) as source:
    picture = source.convert("RGB")
fig, axes = plt.subplots(2, 2, figsize=(10, 7), layout="constrained")
for axis, name in zip(axes.flat, MODEL_NAMES):
    annotated = draw_prediction(picture, predictions[name][example_id], truth[example_id])
    annotated.thumbnail((720, 500))
    axis.imshow(annotated)
    axis.set_title(f"{name} · {example_id}")
    axis.axis("off")
plt.show()

winning_run = next(r for r in results if r["model"] == winners[0])
attempts_path = Path(winning_run["output_dir"]) / "attempts.jsonl"
attempts = [a for a in jsonl_rows(attempts_path) if a["photo_id"] == example_id]
print("Попытки лучшей модели на этом снимке:", len(attempts))
if attempts:
    print(str(attempts[-1].get("raw_text") or attempts[-1].get("raw_record") or attempts[-1].get("error"))[:1200])
print("Итоговая таблица:", run_dir / "comparison.jsonl")